# 4. Model Prediction

This notebook handles loading trained models and making predictions on test soundscapes.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path('/home/kwierman/Desktop/Projects/BirdClef')
MODELS_DIR = BASE_DIR / 'notebooks' / 'models'
TEST_SOUNDSCAPES_DIR = BASE_DIR / 'test_soundscapes'
SAMPLE_SUBMISSION_CSV = BASE_DIR / 'sample_submission.csv'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 4.1 Load Model Configuration

In [ ]:
# Load configuration
with open(MODELS_DIR / 'ensemble_config.json', 'r') as f:
    config = json.load(f)

num_classes = config['num_classes']
species_list = config['species_list']
training_results = config['training_results']

print(f"Number of classes: {num_classes}")
print(f"Species list loaded: {len(species_list)} species")

species_to_idx = {s: i for i, s in enumerate(species_list)}
idx_to_species = {i: s for s, i in species_to_idx.items()}

## 4.2 Define Model Classes

In [ ]:
# CNN Model
class CNNFeatureExtractor(nn.Module):
    def __init__(self, sample_rate=32000):
        super().__init__()
        self.mel_spec = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=512, stride=160, padding=0),
            nn.ReLU(),
        )
        self.conv_layers = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
    
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.mel_spec(x)
        x = self.conv_layers(x)
        return x.squeeze(-1)


class CNNModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = CNNFeatureExtractor()
        self.classifier = nn.Sequential(
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        features = self.features(x)
        return self.classifier(features)


# LSTM Model
class LSTMModel(nn.Module):
    def __init__(self, num_classes, hidden_size=256, num_layers=2):
        super().__init__()
        self.input_proj = nn.Linear(128, 64)
        self.lstm = nn.LSTM(input_size=64, hidden_size=hidden_size, num_layers=num_layers, 
                           batch_first=True, dropout=0.3)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        x = torch.stft(x, n_fft=2048, hop_length=512, return_complex=True)
        x = torch.abs(x).transpose(1, 2)[:, :, :128]
        x = self.input_proj(x)
        _, (h_n, _) = self.lstm(x)
        return self.classifier(h_n[-1])


# ResNet Model
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride),
                nn.BatchNorm1d(out_channels)
            )
    
    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return torch.relu(out)


class ResNet1D(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 64, kernel_size=7, stride=2, padding=3)
        self.bn1 = nn.BatchNorm1d(64)
        self.pool = nn.MaxPool1d(3, stride=2, padding=1)
        self.layer1 = self._make_layer(64, 64, 2, 1)
        self.layer2 = self._make_layer(64, 128, 2, 2)
        self.layer3 = self._make_layer(128, 256, 2, 2)
        self.layer4 = self._make_layer(256, 512, 2, 2)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(512, num_classes)
    
    def _make_layer(self, in_channels, out_channels, blocks, stride):
        layers = [ResidualBlock(in_channels, out_channels, stride)]
        for _ in range(1, blocks):
            layers.append(ResidualBlock(out_channels, out_channels))
        return nn.Sequential(*layers)
    
    def forward(self, x):
        x = x.unsqueeze(1)
        x = torch.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


# Transformer Model
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TransformerModel(nn.Module):
    def __init__(self, num_classes, d_model=256, nhead=8, num_layers=4):
        super().__init__()
        self.input_proj = nn.Linear(128, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, 
                                                    dim_feedforward=512, dropout=0.1, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        x = torch.stft(x, n_fft=2048, hop_length=512, return_complex=True)
        x = torch.abs(x).transpose(1, 2)[:, :, :128]
        x = self.input_proj(x)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = x.mean(dim=1)
        return self.classifier(x)

print("Model classes defined")

## 4.3 Load Trained Models

In [ ]:
def load_model(model_name, num_classes, device):
    """Load a trained model."""
    model_dict = {
        'cnn': CNNModel,
        'lstm': LSTMModel,
        'resnet': ResNet1D,
        'transformer': TransformerModel
    }
    
    model_class = model_dict[model_name]
    model = model_class(num_classes)
    
    # Load weights
    checkpoint_path = MODELS_DIR / f'{model_name}_best.pt'
    if checkpoint_path.exists():
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))
        print(f"Loaded {model_name} from {checkpoint_path}")
    else:
        print(f"Warning: {checkpoint_path} not found")
    
    model = model.to(device)
    model.eval()
    return model

# Load all 4 models
models = {}
for model_name in ['cnn', 'lstm', 'resnet', 'transformer']:
    models[model_name] = load_model(model_name, num_classes, device)

print(f"\nLoaded {len(models)} models for ensemble")

## 4.4 Load Sample Submission

In [ ]:
# Load sample submission to understand structure
sample_sub = pd.read_csv(SAMPLE_SUBMISSION_CSV)
print(f"Sample submission shape: {sample_sub.shape}")
print(f"Number of test files: {sample_sub['row_id'].str.rsplit('_', n=1).str[0].nunique()}")
print(f"\nFirst few row_ids:")
print(sample_sub['row_id'].head())

## 4.5 Audio Loading Utilities

In [ ]:
import torchaudio

def load_audio(file_path, target_sr=32000):
    """Load audio file."""
    waveform, sr = torchaudio.load(file_path)
    if sr != target_sr:
        resampler = torchaudio.transforms.Resample(sr, target_sr)
        waveform = resampler(waveform)
    return waveform

def extract_segment(waveform, start_sec, end_sec, sample_rate=32000):
    """Extract a segment from waveform."""
    start_idx = int(start_sec * sample_rate)
    end_idx = int(end_sec * sample_rate)
    return waveform[:, start_idx:end_idx]

print("Audio utilities defined")

## 4.6 Predict on Test Soundscapes

In [ ]:
def predict_soundscape(file_path, models, device, segment_duration=5):
    """Make predictions for all segments in a soundscape."""
    try:
        waveform = load_audio(file_path)
        
        # Convert to mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        
        # Get total duration
        sample_rate = 32000
        total_duration = waveform.shape[1] / sample_rate
        num_segments = int(total_duration // segment_duration)
        
        predictions = []
        
        for seg_idx in range(num_segments):
            start_sec = seg_idx * segment_duration
            end_sec = start_sec + segment_duration
            
            # Extract segment
            segment = extract_segment(waveform, start_sec, end_sec, sample_rate)
            
            # Ensure correct length
            expected_length = segment_duration * sample_rate
            if segment.shape[1] < expected_length:
                padding = expected_length - segment.shape[1]
                segment = torch.nn.functional.pad(segment, (0, padding))
            
            # Get predictions from all models
            segment_preds = []
            for model in models.values():
                with torch.no_grad():
                    output = model(segment.to(device))
                    pred = torch.sigmoid(output).cpu().numpy().flatten()
                    segment_preds.append(pred)
            
            # Average predictions
            avg_pred = np.mean(segment_preds, axis=0)
            predictions.append(avg_pred)
        
        return np.array(predictions)
        
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

print("Predict function defined")

## 4.7 Process Test Files

In [ ]:
# Check test soundscapes directory
test_files = list(TEST_SOUNDSCAPES_DIR.glob('*.ogg'))
print(f"Found {len(test_files)} test files")

if len(test_files) == 0:
    print("No test files found. Using sample submission structure for demo.")
    print("When test files are available, predictions will be generated.")
    
    # Create predictions based on sample submission structure
    # This shows the expected output format
    print("\nSample submission structure:")
    print(sample_sub.head(2))
    
    # Extract unique test file IDs
    test_file_ids = sample_sub['row_id'].str.rsplit('_', n=1).str[0].unique()
    print(f"\nExpected test files: {len(test_file_ids)}")

In [ ]:
# If test files are available, make predictions
if len(test_files) > 0:
    print(f"Processing {len(test_files)} test files...")
    
    all_predictions = {}
    
    for i, file_path in enumerate(test_files):
        print(f"Processing {i+1}/{len(test_files)}: {file_path.name}")
        predictions = predict_soundscape(str(file_path), models, device)
        
        if predictions is not None:
            all_predictions[file_path.stem] = predictions
    
    print(f"\nProcessed {len(all_predictions)} files")
    print("Predictions ready for submission")
else:
    print("No test files to process - they will be populated at runtime")
    all_predictions = {}

## 4.8 Demo: Predict on Train Soundscapes

In [ ]:
# Demo: Make predictions on a few train soundscapes
TRAIN_SOUNDSCAPES_DIR = BASE_DIR / 'train_soundscapes'
train_sound_files = list(TRAIN_SOUNDSCAPES_DIR.glob('*.ogg'))[:3]

demo_predictions = {}
for file_path in train_sound_files:
    print(f"Processing: {file_path.name}")
    predictions = predict_soundscape(str(file_path), models, device)
    if predictions is not None:
        demo_predictions[file_path.stem] = predictions
        print(f"  Shape: {predictions.shape} (segments, classes)")

print(f"\nDemo predictions for {len(demo_predictions)} files")

## 4.9 Prediction Statistics

In [ ]:
if demo_predictions:
    # Analyze predictions
    all_preds = np.vlist(list(demo_predictions.values()))
    print(f"Prediction statistics:")
    print(f"  Min probability: {all_preds.min():.4f}")
    print(f"  Max probability: {all_preds.max():.4f}")
    print(f"  Mean probability: {all_preds.mean():.4f}")
    print(f"  Std probability: {all_preds.std():.4f}")
    
    # Per-class prediction rates
    mean_preds = all_preds.mean(axis=0)
    high_conf_preds = (mean_preds > 0.5).sum()
    print(f"\n  Classes with >50% average prediction: {high_conf_preds}")
    
    # Top predicted classes
    top_classes = np.argsort(mean_preds)[-10:][::-1]
    print(f"\n  Top 10 predicted classes (by avg probability):")
    for i, cls_idx in enumerate(top_classes):
        print(f"    {i+1}. {idx_to_species[cls_idx]}: {mean_preds[cls_idx]:.4f}")

## 4.10 Save Predictions

In [ ]:
PREDICTIONS_DIR = BASE_DIR / 'notebooks' / 'data' / 'predictions'
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

if demo_predictions:
    # Save demo predictions
    for filename, preds in demo_predictions.items():
        np.save(PREDICTIONS_DIR / f'{filename}_predictions.npy', preds)
    print(f"Saved demo predictions to {PREDICTIONS_DIR}")

# If we have actual test predictions, save them
if all_predictions:
    for filename, preds in all_predictions.items():
        np.save(PREDICTIONS_DIR / f'{filename}_predictions.npy', preds)
    print(f"Saved {len(all_predictions)} test predictions")

print("\n✅ Prediction complete!")